## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-06-14-52-49 +0000,bbc,Workers' rights 'second place' to economic growth,https://www.bbc.co.uk/news/videos/cq369l9r634r...
1,2026-11-06-14-52-18 +0000,nypost,Meghan Markle’s estranged dad Thomas arrested ...,https://nypost.com/2026/11/06/world-news/megha...
2,2026-11-06-14-52-00 +0000,wsj,"Hiring blew past expectations with 165,000 job...",https://www.wsj.com/economy/jobs/october-jobs-...
3,2026-11-06-14-50-14 +0000,nypost,US employers surprisingly add 165K jobs in str...,https://nypost.com/2026/11/06/business/us-empl...
4,2026-11-06-14-48-32 +0000,bbc,Burnham says he was not aware of Brand's 'horr...,https://www.bbc.co.uk/news/articles/cjkgee4yw3...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2673/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
248,trump,33
69,new,22
261,midterm,17
215,market,17
191,up,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
385,2026-11-05-17-20-00 +0000,wsj,President Trump put three justices on the Supr...,https://www.wsj.com/politics/policy/midterm-ro...,107
323,2026-11-05-20-47-17 +0000,nypost,"Tech bosses shouldn’t turn on Trump, he’s the ...",https://nypost.com/2026/11/05/business/tech-bo...,104
274,2026-11-05-22-26-00 +0000,wsj,The Trump administration warned conservative g...,https://www.wsj.com/politics/policy/trump-admi...,95
179,2026-11-06-03-11-54 +0000,nypost,Trump pins US stock market plunge on Democrats...,https://nypost.com/2026/11/05/us-news/trump-pi...,93
382,2026-11-05-17-23-49 +0000,bbc,Farage says he is worried Trump will step up I...,https://www.bbc.co.uk/news/articles/cqz0vz55jp...,92


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
385,107,2026-11-05-17-20-00 +0000,wsj,President Trump put three justices on the Supr...,https://www.wsj.com/politics/policy/midterm-ro...
147,81,2026-11-06-07-36-29 +0000,nypost,Rosie O’Donnell considers return to New York f...,https://nypost.com/2026/11/06/us-news/rosie-od...
384,74,2026-11-05-17-20-00 +0000,wsj,Senate Minority Leader Chuck Schumer was jubil...,https://www.wsj.com/politics/policy/chuck-schu...
308,66,2026-11-05-21-10-00 +0000,cbc,California Gov. Gavin Newsom says he'll commut...,https://www.cbc.ca/news/world/california-newso...
274,62,2026-11-05-22-26-00 +0000,wsj,The Trump administration warned conservative g...,https://www.wsj.com/politics/policy/trump-admi...
323,61,2026-11-05-20-47-17 +0000,nypost,"Tech bosses shouldn’t turn on Trump, he’s the ...",https://nypost.com/2026/11/05/business/tech-bo...
286,61,2026-11-05-21-45-00 +0000,wsj,Los Angeles voters have elected Nithya Raman a...,https://www.wsj.com/politics/elections/democra...
263,54,2026-11-05-22-45-00 +0000,wsj,There is a whiff of postelection optimism abou...,https://www.wsj.com/finance/stocks/why-investo...
306,50,2026-11-05-21-13-53 +0000,chitri,A brutal sell-off for AI stocks sends Wall Str...,https://www.chicagotribune.com/2026/11/05/ai-s...
343,41,2026-11-05-19-43-48 +0000,latimes,"Lopez: She's got political savvy, good timing,...",https://www.latimes.com/california/story/2026-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
